# Data Analytics — Exercise 7.3: Pandas DataFrames & Merging

## Sharleen Guerrero | Date: 5/20/2026
This notebook reads GlobalTrade Ltd.'s order and shipment CSV files into Pandas DataFrames, cleans both datasets, and merges them using all four join types to produce a complete order-fulfilment report.

In [1]:
import pandas as pd
import os 

# Verifying if both CSV files are in the same folder
print(os.listdir('.'))

['.ipynb_checkpoints', 'DA_Exercise_7.1.ipynb', 'DA_Exercise_7.2.ipynb', 'DA_Exercise_7.3.ipynb', 'orders.csv', 'shipments.csv']


## 1. Loading CSV Files
We load orders.csv and shipments.csv from the current folder using pd.read_csv()
Both files must be in the same directory as this notebook.


In [14]:
# Loading the CSV files and converting them to Pandas DataFrame
# df_orders is the DataFrame for orders table
# dr_shipments corressponds to shipments table
#pd._read_csv() opens the file, reads all the data, and converts it into s DataFrame
df_orders = pd.read_csv("orders.csv")
df_shipments = pd.read_csv("shipments.csv")

print("---------------- DataFrames ----------------")
print("\nOrders", df_orders)
print("\nShipments", df_shipments)

print()
print("----------------  Exploratory Analysis ----------------")
print("\n******** Orders Table ********")
print(df_orders.head())  # Showing first 5 rows by default
print()
print(df_orders.shape) # Returns dimensions (rows, columns)
print()
print(df_orders.dtypes)  # Shows datatype of every column, .dtype is only for single column
print()
print(df_orders.isnull().sum()) # Detecting missing values, count() counts present values
print() 

print(" ******** Shipments Table ********")
print(df_shipments.head())  
print()
print(df_shipments.shape) 
print()
print(df_shipments.dtypes)  
print()
print(df_shipments.isnull().sum()) 

---------------- DataFrames ----------------

Orders    order_id  customer          product  quantity  unit_price
0       101      Chen           Laptop         2      899.99
1       102     Patel           Tablet         5      499.50
2       103    Müller          Monitor         1      349.00
3       104   Okonkwo         Keyboard        10       79.99
4       105    Santos          Headset         3      149.99
5       106   Larsson           Webcam         4       89.99
6       107       Kim            Mouse         8       29.99
7       108   O'Brien  Docking Station         1      199.00
8       109     Rossi              SSD         6      119.99
9       110  Nakamura          USB Hub         2       39.99

Shipments    order_id  ship_date carrier      status
0       101  1/10/2024     DHL   Delivered
1       102  1/12/2024   FedEx   Delivered
2       103  1/13/2024     DHL  In Transit
3       104  1/14/2024     UPS   Delivered
4       105  1/15/2024   FedEx     Delayed
5      

## 2. Inspect and Clean Both DataFrames
We calculate order_value, run describe(), and convert ship_date to datetime.

In [22]:
# Adding a new column to Orders DataFrame that calculates the order value
# multiplying all values in quantity by all values in unit price
# When referencing a column, always wrap it in [] with name of DataFrame
df_orders["order_value"] = df_orders['quantity'] * df_orders['unit_price']

print("\n******** Updated Orders Table ********\n")
print(df_orders)
# printing updated dataframe to show new order_value column

# describe() prints numeric summary statistics for all number columns
print("\n--------- Summary Statistics ---------\n")
print(df_orders.describe())
print()

# Converting ship_date column from string to datetime format
df_shipments['ship_date'] = pd.to_datetime(df_shipments['ship_date'])
print(df_shipments.dtypes) # Verifying the conversion worked
# [us] means microseconds 
# It's important to convert dates because if not, it's considered a string and you can't
# perform any date operations on it like finding earliest or latest dates

# Printing total order value across all orders formatted to 2dp
total = df_orders['order_value'].sum()
print(f'\nTotal order value: £{total:,.2f}')


******** Updated Orders Table ********

   order_id  customer          product  quantity  unit_price  order_value
0       101      Chen           Laptop         2      899.99      1799.98
1       102     Patel           Tablet         5      499.50      2497.50
2       103    Müller          Monitor         1      349.00       349.00
3       104   Okonkwo         Keyboard        10       79.99       799.90
4       105    Santos          Headset         3      149.99       449.97
5       106   Larsson           Webcam         4       89.99       359.96
6       107       Kim            Mouse         8       29.99       239.92
7       108   O'Brien  Docking Station         1      199.00       199.00
8       109     Rossi              SSD         6      119.99       719.94
9       110  Nakamura          USB Hub         2       39.99        79.98

--------- Summary Statistics ---------

        order_id   quantity  unit_price  order_value
count   10.00000  10.000000   10.000000    10.00000

## 3. Performing All Four DataFrame Merge Types
We merge df_orders and df_shipments on 'order_id' using inner, left, right, and outer.

In [31]:
# Inner-Merge: Only retrives matching rows
# Merge on the shared key order_id because that column exists in both DataFrames
# inner merge only keeps rows where the order_id exists in BOTH tables
# \ at the end just means table is too wide to fit on one line so it's wrapped
inner_merge = pd.merge(df_orders, df_shipments, on="order_id", how="inner")
print("|INNER MERGE -----------------------------")
print(inner_merge)

# Left-Merge: Retrieves all rows from the left table and only matching rows from the right table
# Whichever table you put first in pd.merge() is left, second is right
# Matching rows based on shared key
left_merge = pd.merge(df_orders, df_shipments, on="order_id", how="left")
print("\n|LEFT MERGE -----------------------------")
print(left_merge)

# Right Merge: Retrieves all rows from the right table and only matching rows from the left table
# df_shipments is right (second), df_orders is left (first)
# order_ids 108, 109, 110 get dropped — they're in orders (left) only
# order_id 111 keeps NaN in order columns — it's in shipments (right) only
right_merge = pd.merge(df_orders, df_shipments, on="order_id", how="right")
print("\n|RIGHT MERGE -----------------------------")
print(right_merge)

# identifying the orphan shipment (NaN in customer column)
# orphan uses Boolean indexing to find rows where customer is NaN
# we check customer because after a right merge, any unmatched row
# will have NaN in all order columns (customer, product, quantity etc)
# .isna() returns True where the value IS NaN (opposite of .notna())
orphan = right_merge[right_merge['customer'].isna()]

# .values converts the Pandas Series to a plain Python list (removes the index number)
# [0] grabs just the first item from that list
# without .values[0] the index number would also print, making it look messy
print(f'Orphan shipment: order_id {orphan["order_id"].values[0]}')

# Outer Merge: Retrieves ALL rows from both tables
# Where there is no match, fills with NaN
# This is a full audit — nothing gets dropped from either table
# order_ids 108, 109, 110 → NaN in shipment columns
# order_id 111 → NaN in order columns
outer_merge = pd.merge(df_orders, df_shipments, on="order_id", how="outer")
print("\n|OUTER MERGE -----------------------------")
print(outer_merge)
print(f'Shape: {outer_merge.shape} | Total NaN values: {outer_merge.isnull().sum().sum()}')

|INNER MERGE -----------------------------
   order_id customer   product  quantity  unit_price  order_value  ship_date  \
0       101     Chen    Laptop         2      899.99      1799.98 2024-01-10   
1       102    Patel    Tablet         5      499.50      2497.50 2024-01-12   
2       103   Müller   Monitor         1      349.00       349.00 2024-01-13   
3       104  Okonkwo  Keyboard        10       79.99       799.90 2024-01-14   
4       105   Santos   Headset         3      149.99       449.97 2024-01-15   
5       106  Larsson    Webcam         4       89.99       359.96 2024-01-16   
6       107      Kim     Mouse         8       29.99       239.92 2024-01-18   

  carrier      status  
0     DHL   Delivered  
1   FedEx   Delivered  
2     DHL  In Transit  
3     UPS   Delivered  
4   FedEx     Delayed  
5     UPS   Delivered  
6     DHL  In Transit  

|LEFT MERGE -----------------------------
   order_id  customer          product  quantity  unit_price  order_value  \
0   

## Merge Results Interpretation

- **Inner Merge (7 rows):** Only kept order_ids 101-107 that existed in both tables. Orders 108, 109, 110 (unshipped) and order_id 111 (orphan shipment) were all dropped.

- **Left Merge (10 rows):** All 10 orders were kept. Orders 108, 109, 110 had NaT/NaN in shipment columns because they have no matching shipment record — these are unshipped orders.

- **Right Merge (8 rows):** All 8 shipments were kept. Order_id 111 had NaN in all order columns because it has no matching order record — this is an orphan shipment that needs investigating.

- **Outer Merge (11 rows):** Every row from both tables was kept — nothing dropped. Orders 108, 109, 110 have NaN in shipment columns and order_id 111 has NaN in order columns. This is a full audit of all data.

## 4. Shipped Flag and Fulfilment Rate
Using the LEFT merged DataFrame we flag each order as shipped or unshipped
and calculate the overall fulfilment rate for the current period.

In [33]:
# notna() returns True where the value is NOT NaN
# if carrier has a value, the order was shipped (True)
# if carrier is NaN, the order was not shipped (False)
left_merge['shipped'] = left_merge['carrier'].notna()

# counting how many orders were shipped vs unshipped
shipped_count = left_merge['shipped'].sum()  # True = 1, False = 0 so sum() counts Trues
total_orders = len(left_merge)  # total number of rows in left merge

# calculating fulfilment rate as a percentage
fulfilment_rate = shipped_count / total_orders * 100

print('---- Shipped Flag per Order ------------------')
print(left_merge[['order_id', 'customer', 'order_value', 'shipped']])

# value_counts() counts how many True and False values exist
print('\n---- Shipped vs Unshipped Count ------------------')
print(left_merge['shipped'].value_counts())

print(f'\nFulfilment rate: {fulfilment_rate:.2f}% ({shipped_count}/{total_orders} orders)')

---- Shipped Flag per Order ------------------
   order_id  customer  order_value  shipped
0       101      Chen      1799.98     True
1       102     Patel      2497.50     True
2       103    Müller       349.00     True
3       104   Okonkwo       799.90     True
4       105    Santos       449.97     True
5       106   Larsson       359.96     True
6       107       Kim       239.92     True
7       108   O'Brien       199.00    False
8       109     Rossi       719.94    False
9       110  Nakamura        79.98    False

---- Shipped vs Unshipped Count ------------------
shipped
True     7
False    3
Name: count, dtype: int64

Fulfilment rate: 70.00% (7/10 orders)


## 5. GroupBy Summary on the Inner Merged DataFrame
We analyse performance by delivery status and by carrier.
The inner merge is used here — it contains only fully matched orders.

In [37]:
# groupby status to see order value broken down by delivery status
# .agg() lets us compute multiple statistics in one call
status_summary = inner_merge.groupby('status')['order_value'].agg(['sum', 'mean', 'count'])
print('---- Order Value by Delivery Status ----')
print(status_summary.round(2))  # round() rounds all values to 2dp

# groupby carrier to see which carrier handled the most value
# sort_values(ascending=False) sorts highest to lowest
carrier_totals = inner_merge.groupby('carrier')['order_value'].sum().sort_values(ascending=False)
print('\n---- Carrier Totals (descending) -------')
print(carrier_totals)

# .index[0] returns the label of the first item after sorting — the top carrier
# .iloc[0] returns the actual value of the first item
print(f'Top carrier: {carrier_totals.index[0]} - £{carrier_totals.iloc[0]:,.2f}')

# units dispatched per delivery status
units_per_status = inner_merge.groupby('status')['quantity'].sum()
print('\n---- Units Dispatched by Status --------')
print(units_per_status)

---- Order Value by Delivery Status ----
                sum     mean  count
status                             
Delayed      449.97   449.97      1
Delivered   5457.34  1364.34      4
In Transit   588.92   294.46      2

---- Carrier Totals (descending) -------
carrier
FedEx    2947.47
DHL      2388.90
UPS      1159.86
Name: order_value, dtype: float64
Top carrier: FedEx - £2,947.47

---- Units Dispatched by Status --------
status
Delayed        3
Delivered     21
In Transit     9
Name: quantity, dtype: int64


## Summary
- The fulfilment rate is **70%** (7 of 10 orders shipped); orders 108, 109, and 110 are still awaiting dispatch.
- **FedEx** is the highest-value carrier at £2,947.47 across 2 shipments; only 1 order (Santos, Headset) is currently delayed.
- The orphan shipment (order_id 111) has no matching order record — this data quality issue should be escalated to the warehouse team.